# Section 4 - Defence Implementation

This notebook implements fixed-attack defence evaluation against the Section 2 adversarial examples.

Defences in this notebook:
- D1: inference-time raw-space winsorising on mutable numeric features
- D2: adversarial training using the fixed attack set from Section 2

Evaluation scope:
- clean accuracy is measured on reconstructed held-out splits using the repo's actual preprocessing
- robustness is evaluated against the same saved attacks from Section 2
- results are descriptive fixed-attack defence results, not adaptive re-attack results
- cross-disease defence effectiveness is not treated as a causal benchmark


---
## 4.0 - Imports and Setup

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
OUTPUT_DIR = ROOT / 'output'
DATA_DIR = ROOT / 'data' / 'processed'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')

PALETTE = {
    'baseline': '#9aa7bd',
    'winsor_1_99': '#1f77b4',
    'winsor_5_95': '#6baed6',
    'adv_train': '#d62728',
}

print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Project root: {ROOT}')
print(f'Output dir  : {OUTPUT_DIR}')
print(f'Data dir    : {DATA_DIR}')


---
## 4.1 - Load Models, Artifacts, and Baselines

In [ ]:
DISEASES = ['diabetes', 'heart', 'stroke']
DISPLAY_NAMES = {
    'diabetes': 'Diabetes RF',
    'heart': 'Heart LR',
    'stroke': 'Stroke LR',
}

DIABETES_FEATURES = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'
]
DIABETES_TARGET = 'Outcome'
DIABETES_LOG_FEATURES = ['Insulin', 'DiabetesPedigreeFunction', 'Age']
DIABETES_WINSORIZE_FEATURES = ['Pregnancies', 'Glucose', 'SkinThickness', 'BMI']

HEART_FEATURES = [
    'Age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal'
]
HEART_TARGET = 'target'
HEART_LOG_FEATURES = ['oldpeak', 'chol']

STROKE_RAW_FEATURES = [
    'gender', 'age', 'hypertension', 'heart_disease', 'ever_married',
    'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status'
]
STROKE_TARGET = 'stroke'
STROKE_MODEL_NUMERICAL = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
STROKE_MODEL_CATEGORICAL = ['gender', 'ever_married', 'work_type', 'smoking_status']
STROKE_LOG_FEATURES = ['avg_glucose_level', 'bmi']

diabetes_model = joblib.load(OUTPUT_DIR / 'diabetes_rf_model.joblib')
diabetes_scaler = joblib.load(OUTPUT_DIR / 'diabetes_rf_scaler.joblib')
heart_model = joblib.load(OUTPUT_DIR / 'heart_disease_lasso_model.joblib')
heart_scaler = joblib.load(OUTPUT_DIR / 'heart_disease_scaler.joblib')
stroke_model = joblib.load(OUTPUT_DIR / 'stroke_model.joblib')
stroke_scaler = joblib.load(OUTPUT_DIR / 'stroke_scaler.joblib')
stroke_encoder = joblib.load(OUTPUT_DIR / 'stroke_encoder.joblib')

with open(OUTPUT_DIR / 'diabetes_winsorize_bounds.json', 'r') as f:
    diabetes_winsorize_bounds = json.load(f)

with open(OUTPUT_DIR / 'clinical_constraints.json', 'r') as f:
    constraints_raw = json.load(f)

with open(OUTPUT_DIR / 'vulnerability_report.json', 'r') as f:
    baseline_report = json.load(f)

artifacts = joblib.load(OUTPUT_DIR / 'adversarial_examples.joblib')

baseline_rows = []
for disease in DISEASES:
    baseline_rows.append({
        'Disease': DISPLAY_NAMES[disease],
        'Attack Method': baseline_report['core_metrics'][disease]['attack_method'],
        'Section 3 ASR (%)': baseline_report['core_metrics'][disease]['asr'],
        'Section 2 Seeds': len(artifacts[disease]['results']),
    })

print(pd.DataFrame(baseline_rows).to_string(index=False))


---
## 4.2 - Raw Preprocessing and Held-Out Splits

These helpers mirror the actual repo preprocessing used by the saved models.
Train/test splits are reconstructed with the same `test_size=0.3` and `random_state=42` used in training.


In [ ]:
def build_constraint_map(disease):
    return {item['name']: item for item in constraints_raw[disease]}


def apply_diabetes_winsorization(df):
    engineered = df.copy()
    for col in DIABETES_WINSORIZE_FEATURES:
        bounds = diabetes_winsorize_bounds[col]
        engineered[col] = engineered[col].clip(lower=bounds['lower'], upper=bounds['upper'])
    return engineered


def diabetes_feature_engineering(df):
    engineered = df[DIABETES_FEATURES].copy()
    engineered = apply_diabetes_winsorization(engineered)
    for col in DIABETES_LOG_FEATURES:
        engineered[col] = np.log1p(engineered[col].astype(float))
    return engineered


def heart_feature_engineering(df):
    engineered = df[HEART_FEATURES].copy()
    for col in HEART_LOG_FEATURES:
        engineered[col] = np.log1p(engineered[col].astype(float))
    return engineered


def stroke_feature_engineering(df, encoder=None, fit_encoder=False):
    if encoder is None and not fit_encoder:
        encoder = stroke_encoder

    engineered = df[STROKE_RAW_FEATURES].copy()
    numeric = engineered[STROKE_MODEL_NUMERICAL].copy()
    for col in STROKE_LOG_FEATURES:
        numeric[col] = np.log1p(numeric[col].astype(float))

    categorical = engineered[STROKE_MODEL_CATEGORICAL].copy()

    if fit_encoder:
        encoder = OneHotEncoder(drop='first', sparse_output=False)
        encoder.fit(categorical)

    encoded = encoder.transform(categorical)
    encoded_names = list(encoder.get_feature_names_out(STROKE_MODEL_CATEGORICAL))
    encoded_df = pd.DataFrame(encoded, columns=encoded_names, index=engineered.index)
    processed = pd.concat([numeric, encoded_df], axis=1)
    return processed, encoder


def preprocess_diabetes_raw(df, scaler=None):
    use_scaler = diabetes_scaler if scaler is None else scaler
    return use_scaler.transform(diabetes_feature_engineering(df))


def preprocess_heart_raw(df, scaler=None):
    use_scaler = heart_scaler if scaler is None else scaler
    return use_scaler.transform(heart_feature_engineering(df))


def preprocess_stroke_raw(df, scaler=None, encoder=None):
    use_scaler = stroke_scaler if scaler is None else scaler
    use_encoder = stroke_encoder if encoder is None else encoder
    processed, _ = stroke_feature_engineering(df, encoder=use_encoder, fit_encoder=False)
    return use_scaler.transform(processed)


def attack_results_to_raw_df(results, feature_order, key):
    return pd.DataFrame([row[key] for row in results])[feature_order].reset_index(drop=True)


diabetes_df = pd.read_csv(DATA_DIR / 'diabetes_processed.csv')
heart_df = pd.read_csv(DATA_DIR / 'heart_processed.csv')
stroke_df = pd.read_csv(DATA_DIR / 'stroke_processed.csv').drop(columns=['id'], errors='ignore')

X_dia_train_raw, X_dia_test_raw, y_dia_train, y_dia_test = train_test_split(
    diabetes_df[DIABETES_FEATURES],
    diabetes_df[DIABETES_TARGET],
    test_size=0.3,
    random_state=42,
)

X_heart_train_raw, X_heart_test_raw, y_heart_train, y_heart_test = train_test_split(
    heart_df[HEART_FEATURES],
    heart_df[HEART_TARGET],
    test_size=0.3,
    random_state=42,
)

X_stroke_train_raw, X_stroke_test_raw, y_stroke_train, y_stroke_test = train_test_split(
    stroke_df[STROKE_RAW_FEATURES],
    stroke_df[STROKE_TARGET],
    test_size=0.3,
    random_state=42,
)

split_df = pd.DataFrame([
    {'Disease': 'Diabetes', 'Train': len(X_dia_train_raw), 'Test': len(X_dia_test_raw)},
    {'Disease': 'Heart', 'Train': len(X_heart_train_raw), 'Test': len(X_heart_test_raw)},
    {'Disease': 'Stroke', 'Train': len(X_stroke_train_raw), 'Test': len(X_stroke_test_raw)},
])
print(split_df.to_string(index=False))


---
## 4.3 - Baseline Clean Accuracy

These are clean-data baselines for the saved models on the reconstructed held-out splits.


In [ ]:
def evaluate_clean_accuracy(model, X_test_raw, y_test, preprocess_fn, preprocess_kwargs=None):
    kwargs = {} if preprocess_kwargs is None else preprocess_kwargs
    preds = model.predict(preprocess_fn(X_test_raw, **kwargs))
    y_true = np.asarray(y_test).astype(int)
    y_pred = np.asarray(preds).astype(int)
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        zero_division=0,
    )
    return {
        'accuracy': round(float(acc * 100), 2),
        'precision': round(float(precision * 100), 2),
        'recall': round(float(recall * 100), 2),
        'f1': round(float(f1 * 100), 2),
    }


baseline_clean = {
    'diabetes': evaluate_clean_accuracy(diabetes_model, X_dia_test_raw, y_dia_test, preprocess_diabetes_raw),
    'heart': evaluate_clean_accuracy(heart_model, X_heart_test_raw, y_heart_test, preprocess_heart_raw),
    'stroke': evaluate_clean_accuracy(stroke_model, X_stroke_test_raw, y_stroke_test, preprocess_stroke_raw),
}

baseline_clean_df = pd.DataFrame([
    {
        'Disease': DISPLAY_NAMES[disease],
        'Clean Accuracy (%)': baseline_clean[disease]['accuracy'],
        'Precision (%)': baseline_clean[disease]['precision'],
        'Recall (%)': baseline_clean[disease]['recall'],
        'F1 (%)': baseline_clean[disease]['f1'],
        'Section 3 ASR (%)': baseline_report['core_metrics'][disease]['asr'],
    }
    for disease in DISEASES
])

print(baseline_clean_df.to_string(index=False))


---
## 4.4 - Defence 1: Inference-Time Winsorising

This defence clips mutable numeric raw features to percentile bounds learned from the training split.
It is evaluated as a fixed-attack defence against the saved Section 2 adversarial examples.


In [ ]:
class RawWinsorisingDefence:
    def __init__(self, model, preprocess_fn, feature_order, mutable_numeric_features, preprocess_kwargs=None, p_low=1.0, p_high=99.0):
        self.model = model
        self.preprocess_fn = preprocess_fn
        self.feature_order = feature_order
        self.mutable_numeric_features = list(mutable_numeric_features)
        self.preprocess_kwargs = {} if preprocess_kwargs is None else preprocess_kwargs
        self.p_low = p_low
        self.p_high = p_high
        self.bounds_ = {}

    def fit(self, X_train_raw):
        for feature in self.mutable_numeric_features:
            series = pd.to_numeric(X_train_raw[feature], errors='coerce')
            self.bounds_[feature] = {
                'lower': float(np.nanpercentile(series, self.p_low)),
                'upper': float(np.nanpercentile(series, self.p_high)),
            }
        return self

    def sanitise_df(self, df):
        sanitised = df[self.feature_order].copy()
        for feature, bounds in self.bounds_.items():
            sanitised[feature] = pd.to_numeric(sanitised[feature], errors='coerce').clip(bounds['lower'], bounds['upper'])
        return sanitised

    def predict(self, df):
        sanitised = self.sanitise_df(df)
        X_model = self.preprocess_fn(sanitised, **self.preprocess_kwargs)
        return self.model.predict(X_model)


def mutable_numeric_features_for(disease, feature_order):
    constraint_map = build_constraint_map(disease)
    selected = []
    for feature in feature_order:
        item = constraint_map.get(feature)
        if item is None:
            continue
        if item.get('mutable', True) and not item.get('is_categorical', False):
            selected.append(feature)
    return selected


def evaluate_fixed_attack_defence(results, feature_order, predict_fn):
    successful_before = [row for row in results if row['success']]
    still_flip = 0
    for row in successful_before:
        candidate_df = pd.DataFrame([row['x_adv']])[feature_order]
        pred = int(predict_fn(candidate_df)[0])
        if pred == 0:
            still_flip += 1
    total = len(results)
    residual_asr = 100.0 * still_flip / total if total else 0.0
    residual_on_original_successes = 100.0 * still_flip / len(successful_before) if successful_before else 0.0
    return {
        'n_original_successes': len(successful_before),
        'n_still_flip': still_flip,
        'defended_asr': round(float(residual_asr), 2),
        'residual_on_original_successes': round(float(residual_on_original_successes), 2),
    }


def evaluate_winsorising_defence(disease, model, preprocess_fn, feature_order, X_train_raw, X_test_raw, y_test, results, preprocess_kwargs=None):
    disease_results = {}
    mutable_numeric = mutable_numeric_features_for(disease, feature_order)
    for p_low, p_high in [(1, 99), (5, 95)]:
        defence = RawWinsorisingDefence(
            model=model,
            preprocess_fn=preprocess_fn,
            feature_order=feature_order,
            mutable_numeric_features=mutable_numeric,
            preprocess_kwargs=preprocess_kwargs,
            p_low=p_low,
            p_high=p_high,
        ).fit(X_train_raw)
        clean_preds = defence.predict(X_test_raw)
        clean_acc = 100.0 * accuracy_score(y_test, clean_preds)
        attack_eval = evaluate_fixed_attack_defence(results, feature_order, defence.predict)
        disease_results[f'p{p_low}_{p_high}'] = {
            'defence_name': f'Winsorising [{p_low}, {p_high}]',
            'clean_acc': round(float(clean_acc), 2),
            'mutable_numeric_features': mutable_numeric,
            'bounds': defence.bounds_,
            'defence_object': defence,
            **attack_eval,
        }
    return disease_results


winsor_results = {
    'diabetes': evaluate_winsorising_defence('diabetes', diabetes_model, preprocess_diabetes_raw, DIABETES_FEATURES, X_dia_train_raw, X_dia_test_raw, y_dia_test, artifacts['diabetes']['results']),
    'heart': evaluate_winsorising_defence('heart', heart_model, preprocess_heart_raw, HEART_FEATURES, X_heart_train_raw, X_heart_test_raw, y_heart_test, artifacts['heart']['results']),
    'stroke': evaluate_winsorising_defence('stroke', stroke_model, preprocess_stroke_raw, STROKE_RAW_FEATURES, X_stroke_train_raw, X_stroke_test_raw, y_stroke_test, artifacts['stroke']['results']),
}

winsor_rows = []
for disease in DISEASES:
    baseline_asr = baseline_report['core_metrics'][disease]['asr']
    baseline_acc_val = baseline_clean[disease]['accuracy']
    for key in ['p1_99', 'p5_95']:
        result = winsor_results[disease][key]
        winsor_rows.append({
            'Disease': DISPLAY_NAMES[disease],
            'Defence': result['defence_name'],
            'Baseline ASR (%)': baseline_asr,
            'Defended ASR (%)': result['defended_asr'],
            'ASR Reduction (pp)': round(float(baseline_asr - result['defended_asr']), 2),
            'Baseline Clean Acc (%)': baseline_acc_val,
            'Defended Clean Acc (%)': result['clean_acc'],
            'Residual on Orig Successes (%)': result['residual_on_original_successes'],
        })

winsor_df = pd.DataFrame(winsor_rows)
print(winsor_df.to_string(index=False))


---
## 4.5 - Defence 2: Adversarial Training

This defence augments the original training split with the fixed Section 2 adversarial examples, labelled with their true class `1`, then retrains using the original pipeline family for each disease.


In [ ]:
def predict_with_bundle(bundle, df):
    disease = bundle['disease']
    if disease == 'diabetes':
        features = diabetes_feature_engineering(df)
    elif disease == 'heart':
        features = heart_feature_engineering(df)
    elif disease == 'stroke':
        features, _ = stroke_feature_engineering(df, encoder=bundle['encoder'], fit_encoder=False)
    else:
        raise ValueError(f'Unsupported disease bundle: {disease}')
    X_scaled = bundle['scaler'].transform(features)
    return bundle['model'].predict(X_scaled)


def train_diabetes_adv_bundle(X_train_raw, y_train, adv_results):
    X_adv_raw = attack_results_to_raw_df(adv_results, DIABETES_FEATURES, 'x_adv')
    y_adv = np.ones(len(X_adv_raw), dtype=int)
    X_aug_raw = pd.concat([X_train_raw.reset_index(drop=True), X_adv_raw], ignore_index=True)
    y_aug = np.concatenate([np.asarray(y_train).astype(int), y_adv])
    X_aug_features = diabetes_feature_engineering(X_aug_raw)
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_aug_features, y_aug)
    scaler = StandardScaler()
    X_resampled_scaled = scaler.fit_transform(X_resampled)
    model = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
    model.fit(X_resampled_scaled, y_resampled)
    return {
        'disease': 'diabetes',
        'model': model,
        'scaler': scaler,
        'n_augmented': int(len(X_adv_raw)),
    }


def train_heart_adv_bundle(X_train_raw, y_train, adv_results):
    X_adv_raw = attack_results_to_raw_df(adv_results, HEART_FEATURES, 'x_adv')
    y_adv = np.ones(len(X_adv_raw), dtype=int)
    X_aug_raw = pd.concat([X_train_raw.reset_index(drop=True), X_adv_raw], ignore_index=True)
    y_aug = np.concatenate([np.asarray(y_train).astype(int), y_adv])
    X_aug_features = heart_feature_engineering(X_aug_raw)
    scaler = StandardScaler()
    X_aug_scaled = scaler.fit_transform(X_aug_features)
    model = LogisticRegression(penalty='l1', max_iter=5000, C=0.1, solver='saga', random_state=42)
    model.fit(X_aug_scaled, y_aug)
    return {
        'disease': 'heart',
        'model': model,
        'scaler': scaler,
        'n_augmented': int(len(X_adv_raw)),
    }


def train_stroke_adv_bundle(X_train_raw, y_train, adv_results):
    X_adv_raw = attack_results_to_raw_df(adv_results, STROKE_RAW_FEATURES, 'x_adv')
    y_adv = np.ones(len(X_adv_raw), dtype=int)
    X_aug_raw = pd.concat([X_train_raw.reset_index(drop=True), X_adv_raw], ignore_index=True)
    y_aug = np.concatenate([np.asarray(y_train).astype(int), y_adv])
    X_aug_features, encoder = stroke_feature_engineering(X_aug_raw, fit_encoder=True)
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_aug_features, y_aug)
    scaler = StandardScaler()
    X_resampled_scaled = scaler.fit_transform(X_resampled)
    model = LogisticRegression(penalty='l1', C=0.01, solver='liblinear', max_iter=5000, random_state=42)
    model.fit(X_resampled_scaled, y_resampled)
    return {
        'disease': 'stroke',
        'model': model,
        'scaler': scaler,
        'encoder': encoder,
        'n_augmented': int(len(X_adv_raw)),
    }


def evaluate_adv_training_bundle(bundle, X_test_raw, y_test, results, feature_order):
    clean_preds = predict_with_bundle(bundle, X_test_raw)
    clean_acc = 100.0 * accuracy_score(y_test, clean_preds)
    attack_eval = evaluate_fixed_attack_defence(results, feature_order, lambda df: predict_with_bundle(bundle, df))
    return {
        'clean_acc': round(float(clean_acc), 2),
        **attack_eval,
    }


advtrain_bundles = {
    'diabetes': train_diabetes_adv_bundle(X_dia_train_raw, y_dia_train, artifacts['diabetes']['results']),
    'heart': train_heart_adv_bundle(X_heart_train_raw, y_heart_train, artifacts['heart']['results']),
    'stroke': train_stroke_adv_bundle(X_stroke_train_raw, y_stroke_train, artifacts['stroke']['results']),
}

advtrain_results = {
    'diabetes': evaluate_adv_training_bundle(advtrain_bundles['diabetes'], X_dia_test_raw, y_dia_test, artifacts['diabetes']['results'], DIABETES_FEATURES),
    'heart': evaluate_adv_training_bundle(advtrain_bundles['heart'], X_heart_test_raw, y_heart_test, artifacts['heart']['results'], HEART_FEATURES),
    'stroke': evaluate_adv_training_bundle(advtrain_bundles['stroke'], X_stroke_test_raw, y_stroke_test, artifacts['stroke']['results'], STROKE_RAW_FEATURES),
}

adv_rows = []
for disease in DISEASES:
    baseline_asr = baseline_report['core_metrics'][disease]['asr']
    baseline_acc_val = baseline_clean[disease]['accuracy']
    result = advtrain_results[disease]
    adv_rows.append({
        'Disease': DISPLAY_NAMES[disease],
        'Defence': 'Adversarial Training',
        'Baseline ASR (%)': baseline_asr,
        'Defended ASR (%)': result['defended_asr'],
        'ASR Reduction (pp)': round(float(baseline_asr - result['defended_asr']), 2),
        'Baseline Clean Acc (%)': baseline_acc_val,
        'Defended Clean Acc (%)': result['clean_acc'],
        'Residual on Orig Successes (%)': result['residual_on_original_successes'],
        'Augmented Rows': advtrain_bundles[disease]['n_augmented'],
    })

advtrain_df = pd.DataFrame(adv_rows)
print(advtrain_df.to_string(index=False))


---
## 4.6 - Defence Comparison and Section 5 Bridge

The comparison table reports defence effectiveness within each disease setup.
The seed-retention check shows how many original high-risk seeds remain high-risk under each defence, which matters for Section 5 explanation analysis.


In [ ]:
def tradeoff_score(asr_reduction, acc_cost):
    if acc_cost <= 0.01:
        return 'inf' if asr_reduction > 0 else 'n/a'
    return f'{asr_reduction / acc_cost:.2f}'


comparison_rows = []
for disease in DISEASES:
    baseline_asr = baseline_report['core_metrics'][disease]['asr']
    baseline_acc_val = baseline_clean[disease]['accuracy']
    for key in ['p1_99', 'p5_95']:
        result = winsor_results[disease][key]
        asr_reduction = round(float(baseline_asr - result['defended_asr']), 2)
        acc_cost = round(float(baseline_acc_val - result['clean_acc']), 2)
        comparison_rows.append({
            'Disease': DISPLAY_NAMES[disease],
            'Defence': result['defence_name'],
            'Baseline ASR (%)': baseline_asr,
            'Defended ASR (%)': result['defended_asr'],
            'ASR Reduction (pp)': asr_reduction,
            'Baseline Clean Acc (%)': baseline_acc_val,
            'Defended Clean Acc (%)': result['clean_acc'],
            'Acc Cost (pp)': acc_cost,
            'Trade-off Score': tradeoff_score(asr_reduction, acc_cost),
        })
    adv_result = advtrain_results[disease]
    adv_asr_reduction = round(float(baseline_asr - adv_result['defended_asr']), 2)
    adv_acc_cost = round(float(baseline_acc_val - adv_result['clean_acc']), 2)
    comparison_rows.append({
        'Disease': DISPLAY_NAMES[disease],
        'Defence': 'Adversarial Training',
        'Baseline ASR (%)': baseline_asr,
        'Defended ASR (%)': adv_result['defended_asr'],
        'ASR Reduction (pp)': adv_asr_reduction,
        'Baseline Clean Acc (%)': baseline_acc_val,
        'Defended Clean Acc (%)': adv_result['clean_acc'],
        'Acc Cost (pp)': adv_acc_cost,
        'Trade-off Score': tradeoff_score(adv_asr_reduction, adv_acc_cost),
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))

seed_retention_rows = []
for disease, feature_order in [
    ('diabetes', DIABETES_FEATURES),
    ('heart', HEART_FEATURES),
    ('stroke', STROKE_RAW_FEATURES),
]:
    orig_seed_df = attack_results_to_raw_df(artifacts[disease]['results'], feature_order, 'x_orig')
    winsor_pred = winsor_results[disease]['p1_99']['defence_object'].predict(orig_seed_df)
    adv_pred = predict_with_bundle(advtrain_bundles[disease], orig_seed_df)
    seed_retention_rows.append({
        'Disease': DISPLAY_NAMES[disease],
        'Winsorising [1, 99] high-risk seeds kept': int(np.sum(winsor_pred == 1)),
        'Adv Training high-risk seeds kept': int(np.sum(adv_pred == 1)),
        'Total seeds': int(len(orig_seed_df)),
    })

seed_retention_df = pd.DataFrame(seed_retention_rows)
print('\nSeed retention for Section 5 bridge:')
print(seed_retention_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
active_diseases = DISEASES
x = np.arange(len(active_diseases))
width = 0.2
labels = [DISPLAY_NAMES[disease] for disease in active_diseases]

baseline_asr_vals = [baseline_report['core_metrics'][disease]['asr'] for disease in active_diseases]
winsor_1_asr_vals = [winsor_results[disease]['p1_99']['defended_asr'] for disease in active_diseases]
winsor_5_asr_vals = [winsor_results[disease]['p5_95']['defended_asr'] for disease in active_diseases]
adv_asr_vals = [advtrain_results[disease]['defended_asr'] for disease in active_diseases]

axes[0].bar(x - 1.5 * width, baseline_asr_vals, width, label='Baseline', color=PALETTE['baseline'])
axes[0].bar(x - 0.5 * width, winsor_1_asr_vals, width, label='Winsor [1,99]', color=PALETTE['winsor_1_99'])
axes[0].bar(x + 0.5 * width, winsor_5_asr_vals, width, label='Winsor [5,95]', color=PALETTE['winsor_5_95'])
axes[0].bar(x + 1.5 * width, adv_asr_vals, width, label='Adv Train', color=PALETTE['adv_train'])
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_ylabel('ASR (%)')
axes[0].set_title('A. Fixed-attack ASR by defence')
axes[0].legend(fontsize=8)

baseline_acc_vals = [baseline_clean[disease]['accuracy'] for disease in active_diseases]
winsor_1_acc_vals = [winsor_results[disease]['p1_99']['clean_acc'] for disease in active_diseases]
winsor_5_acc_vals = [winsor_results[disease]['p5_95']['clean_acc'] for disease in active_diseases]
adv_acc_vals = [advtrain_results[disease]['clean_acc'] for disease in active_diseases]

axes[1].bar(x - 1.5 * width, baseline_acc_vals, width, label='Baseline', color=PALETTE['baseline'])
axes[1].bar(x - 0.5 * width, winsor_1_acc_vals, width, label='Winsor [1,99]', color=PALETTE['winsor_1_99'])
axes[1].bar(x + 0.5 * width, winsor_5_acc_vals, width, label='Winsor [5,95]', color=PALETTE['winsor_5_95'])
axes[1].bar(x + 1.5 * width, adv_acc_vals, width, label='Adv Train', color=PALETTE['adv_train'])
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylabel('Clean accuracy (%)')
axes[1].set_title('B. Clean accuracy by defence')

for disease in active_diseases:
    disease_rows = comparison_df[comparison_df['Disease'] == DISPLAY_NAMES[disease]]
    for _, row in disease_rows.iterrows():
        if row['Defence'] == 'Winsorising [1, 99]':
            color = PALETTE['winsor_1_99']
        elif row['Defence'] == 'Winsorising [5, 95]':
            color = PALETTE['winsor_5_95']
        else:
            color = PALETTE['adv_train']
        axes[2].scatter(row['Acc Cost (pp)'], row['ASR Reduction (pp)'], color=color, s=120)
        axes[2].annotate(f"{disease[:3]}\n{row['Defence'].replace('Winsorising ', 'W ')}", (row['Acc Cost (pp)'], row['ASR Reduction (pp)']), textcoords='offset points', xytext=(4, 3), fontsize=7)

axes[2].axhline(0, color='gray', linestyle='--', alpha=0.4)
axes[2].axvline(0, color='gray', linestyle='--', alpha=0.4)
axes[2].set_xlabel('Clean accuracy cost (pp)')
axes[2].set_ylabel('ASR reduction (pp)')
axes[2].set_title('C. Robustness-accuracy trade-off')

plt.suptitle('Section 4 Defence Evaluation (fixed attacks, within-setup view)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sec4_defence_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved figure -> {OUTPUT_DIR / 'sec4_defence_comparison.png'}")


---
## 4.7 - Save Defence Artifacts

In [ ]:
bundle_paths = {}
for disease in DISEASES:
    path = OUTPUT_DIR / f'{disease}_adv_trained_bundle.joblib'
    joblib.dump(advtrain_bundles[disease], path)
    bundle_paths[disease] = str(path)
    print(f'Saved bundle -> {path.name}')

defence_report = {
    'notes': [
        'Defence evaluation uses fixed Section 2 attacks and does not include adaptive re-attacks.',
        'Clean accuracy is evaluated on reconstructed held-out splits with repo-aligned preprocessing.',
        'Cross-disease defence comparisons remain descriptive rather than causal benchmarks.',
    ],
    'baseline_clean_metrics': baseline_clean,
    'section3_reference': baseline_report['core_metrics'],
    'winsorising': {},
    'adversarial_training': {},
    'seed_retention': seed_retention_df.to_dict(orient='records'),
    'saved_bundles': bundle_paths,
}

for disease in DISEASES:
    defence_report['winsorising'][disease] = {}
    for key in ['p1_99', 'p5_95']:
        result = winsor_results[disease][key]
        defence_report['winsorising'][disease][key] = {
            'defence_name': result['defence_name'],
            'clean_acc': result['clean_acc'],
            'defended_asr': result['defended_asr'],
            'n_still_flip': result['n_still_flip'],
            'n_original_successes': result['n_original_successes'],
            'residual_on_original_successes': result['residual_on_original_successes'],
            'mutable_numeric_features': result['mutable_numeric_features'],
            'bounds': result['bounds'],
        }
    defence_report['adversarial_training'][disease] = {
        'clean_acc': advtrain_results[disease]['clean_acc'],
        'defended_asr': advtrain_results[disease]['defended_asr'],
        'n_still_flip': advtrain_results[disease]['n_still_flip'],
        'n_original_successes': advtrain_results[disease]['n_original_successes'],
        'residual_on_original_successes': advtrain_results[disease]['residual_on_original_successes'],
        'n_augmented': advtrain_bundles[disease]['n_augmented'],
    }

with open(OUTPUT_DIR / 'defence_report.json', 'w') as f:
    json.dump(defence_report, f, indent=2)

print(f"Saved report -> {OUTPUT_DIR / 'defence_report.json'}")


---
## 4.8 - Section Summary

In [ ]:
print('=' * 72)
print('SECTION 4 SUMMARY - DEFENCE IMPLEMENTATION')
print('=' * 72)

for disease in DISEASES:
    baseline_asr = baseline_report['core_metrics'][disease]['asr']
    best_row = comparison_df[comparison_df['Disease'] == DISPLAY_NAMES[disease]].sort_values('Defended ASR (%)').iloc[0]
    print(f"\n{DISPLAY_NAMES[disease]}")
    print(f"  Baseline ASR       : {baseline_asr:.1f}%")
    print(f"  Baseline clean acc : {baseline_clean[disease]['accuracy']:.2f}%")
    print(f"  Lowest defended ASR: {best_row['Defended ASR (%)']:.1f}% via {best_row['Defence']}")
    print(f"  Defended clean acc : {best_row['Defended Clean Acc (%)']:.2f}%")

print('\nKey caveats:')
print('- This is a fixed-attack defence evaluation, not an adaptive robustness benchmark.')
print('- Results are intended as Section 5 baseline context, not as a final defence claim.')
print('- Cross-disease differences remain descriptive because the models and attack surfaces differ.')

print('\nSaved outputs:')
print(f"- {OUTPUT_DIR / 'defence_report.json'}")
print(f"- {OUTPUT_DIR / 'sec4_defence_comparison.png'}")
for disease in DISEASES:
    print(f"- {OUTPUT_DIR / (disease + '_adv_trained_bundle.joblib')}")
